# Muhtemel Ask - 01 PREPARE
Run all cells in a Google Colab GPU runtime. Ordinary use requires changing only `EPISODE` and `SOURCE_URL`. Completed stages are hash-validated and resumed from Google Drive.

In [ ]:
EPISODE = 12  # @param {type:"integer"}
SOURCE_URL = ""  # @param {type:"string"}

if EPISODE < 1:
    raise ValueError("EPISODE must be a positive integer")
if not SOURCE_URL.startswith(("https://www.youtube.com/", "https://youtu.be/", "https://www.dailymotion.com/", "https://dai.ly/")):
    raise ValueError("Set SOURCE_URL to the permitted YouTube or Dailymotion source")

In [ ]:
# Advanced settings - normal use does not require changes.
SCHEMA_VERSION = "1.0"  # @param {type:"string"}
WHISPER_MODEL = "large-v3"  # @param {type:"string"}
BATCH_SIZE = 400  # @param {type:"integer"}
TARGETED_VERIFICATION = True  # @param {type:"boolean"}
FORCE_REBUILD = False  # @param {type:"boolean"}

if not 350 <= BATCH_SIZE <= 500:
    raise ValueError("BATCH_SIZE must be between 350 and 500")

## Mount Drive and install the Colab runtime dependencies

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import os
import shutil
import subprocess
import sys

SYSTEM_ROOT = Path("/content/drive/MyDrive/Muhtemel_Ask_Subtitles/SYSTEM")
if not (SYSTEM_ROOT / "src").is_dir():
    raise FileNotFoundError(
        f"System files were not found at {SYSTEM_ROOT}. Follow 00_README_FIRST.md."
    )
if shutil.which("ffmpeg") is None or shutil.which("ffprobe") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)
if shutil.which("deno") is None:
    deno_installer = Path("/tmp/deno-install.sh")
    subprocess.run(
        ["curl", "-fsSL", "https://deno.land/install.sh", "-o", str(deno_installer)],
        check=True,
    )
    deno_env = dict(os.environ)
    deno_env["DENO_INSTALL"] = "/usr/local"
    subprocess.run(["sh", str(deno_installer)], check=True, env=deno_env)
if shutil.which("deno") is None:
    raise RuntimeError("Deno installation failed; yt-dlp requires a supported JS runtime")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(SYSTEM_ROOT / "requirements-colab.txt")],
    check=True,
)
if str(SYSTEM_ROOT) not in sys.path:
    sys.path.insert(0, str(SYSTEM_ROOT))
print("Runtime ready.")

## Create the isolated episode workspace

In [ ]:
import yaml

with (SYSTEM_ROOT / "config/series.yaml").open(encoding="utf-8") as f:
    series_config = yaml.safe_load(f)
with (SYSTEM_ROOT / "config/names.yaml").open(encoding="utf-8") as f:
    names_config = yaml.safe_load(f)
with (SYSTEM_ROOT / "config/religious_terms.yaml").open(encoding="utf-8") as f:
    religious_config = yaml.safe_load(f)

EPISODE_NAME = f"Muhtemel Ask {EPISODE}.Bolum"
EPISODE_ROOT = Path(series_config["drive_root"]) / "EPISODES" / EPISODE_NAME
DIRS = {name: EPISODE_ROOT / name for name in (
    "source", "prepare", "translation_input",
    "translation_output", "review", "final"
)}
for folder in DIRS.values():
    folder.mkdir(parents=True, exist_ok=True)
print(f"Episode workspace: {EPISODE_ROOT}")

## Reusable YouTube authentication

In [ ]:
YOUTUBE_COOKIE_DRIVE_PATH = (
    Path(series_config["drive_root"]) / "PRIVATE" / "youtube-cookies.txt"
)
YOUTUBE_COOKIE_RUNTIME_ROOT = Path("/content")


def normalise_and_validate_youtube_cookie_payload(payload):
    """Return safe Netscape cookie bytes containing YouTube domains only."""
    import http.cookiejar
    import tempfile
    import warnings

    payload = bytes(payload)
    if not 0 < len(payload) <= 5 * 1024 * 1024:
        raise ValueError("The cookie file must be between 1 byte and 5 MiB")
    payload = payload.removeprefix(b"\xef\xbb\xbf")
    payload = payload.replace(b"\r\n", b"\n").replace(b"\r", b"\n")
    first_line = payload.split(b"\n", 1)[0]
    if first_line not in {b"# HTTP Cookie File", b"# Netscape HTTP Cookie File"}:
        raise ValueError("Use a Netscape-format cookies.txt file")
    check_dir = Path(tempfile.mkdtemp(
        prefix=".muhtemel-ask-cookie-check-", dir=str(YOUTUBE_COOKIE_RUNTIME_ROOT)
    ))
    os.chmod(check_dir, 0o700)
    check_path = check_dir / "cookies.txt"
    try:
        check_path.write_bytes(payload)
        os.chmod(check_path, 0o600)
        jar = http.cookiejar.MozillaCookieJar(str(check_path))
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", UserWarning)
                jar.load(ignore_discard=True, ignore_expires=True)
        except (OSError, UnicodeError, http.cookiejar.LoadError):
            raise ValueError("The cookies.txt file is invalid") from None
        cookies = list(jar)
        if not cookies:
            raise ValueError("The cookies.txt file contains no cookies")
        domains = {cookie.domain.lstrip(".").casefold() for cookie in cookies}
        if not all(domain == "youtube.com" or domain.endswith(".youtube.com") for domain in domains):
            raise ValueError("Export only youtube.com cookies, not all browser cookies")
        return payload
    finally:
        shutil.rmtree(check_dir, ignore_errors=True)


def stage_youtube_cookies(payload):
    """Create the only cookie copy that yt-dlp is allowed to read."""
    import tempfile

    payload = normalise_and_validate_youtube_cookie_payload(payload)
    auth_dir = Path(tempfile.mkdtemp(
        prefix=".muhtemel-ask-youtube-", dir=str(YOUTUBE_COOKIE_RUNTIME_ROOT)
    ))
    os.chmod(auth_dir, 0o700)
    target = auth_dir / "cookies.txt"
    try:
        target.write_bytes(payload)
        os.chmod(target, 0o600)
        return target
    except BaseException:
        shutil.rmtree(auth_dir, ignore_errors=True)
        raise


def load_saved_youtube_cookies():
    """Copy the saved Drive credential into the private runtime."""
    if not YOUTUBE_COOKIE_DRIVE_PATH.is_file():
        return None
    try:
        cookie_path = stage_youtube_cookies(YOUTUBE_COOKIE_DRIVE_PATH.read_bytes())
    except (OSError, ValueError):
        print("Saved YouTube cookies are unreadable or invalid; upload a fresh file.")
        return None
    print("Saved YouTube cookies loaded from Drive.")
    return cookie_path


def save_youtube_cookies_to_drive(cookie_path):
    """Atomically replace the reusable Drive copy after validation."""
    import tempfile

    try:
        payload = normalise_and_validate_youtube_cookie_payload(Path(cookie_path).read_bytes())
        YOUTUBE_COOKIE_DRIVE_PATH.parent.mkdir(parents=True, exist_ok=True)
        descriptor, temporary_name = tempfile.mkstemp(
            prefix=".youtube-cookies.", suffix=".tmp",
            dir=str(YOUTUBE_COOKIE_DRIVE_PATH.parent),
        )
        temporary_path = Path(temporary_name)
        try:
            with os.fdopen(descriptor, "wb") as handle:
                handle.write(payload)
                handle.flush()
            os.replace(temporary_path, YOUTUBE_COOKIE_DRIVE_PATH)
        except BaseException:
            temporary_path.unlink(missing_ok=True)
            raise
    except (OSError, ValueError):
        print("WARNING: YouTube cookies could not be saved to Drive.")
        return False
    print("YouTube cookies saved to Drive for later runs.")
    return True


def upload_youtube_cookies():
    """Upload one YouTube-only Netscape cookie file into this runtime."""
    from google.colab import files
    import tempfile

    upload_dir = Path(tempfile.mkdtemp(
        prefix=".muhtemel-ask-upload-", dir=str(YOUTUBE_COOKIE_RUNTIME_ROOT)
    ))
    os.chmod(upload_dir, 0o700)
    previous_cwd = Path.cwd()
    uploaded = {}
    try:
        os.chdir(upload_dir)
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError("Upload exactly one YouTube cookies.txt file")
        cookie_path = stage_youtube_cookies(next(iter(uploaded.values())))
        print("YouTube cookies loaded into the temporary Colab runtime.")
        return cookie_path
    finally:
        os.chdir(previous_cwd)
        uploaded.clear()
        shutil.rmtree(upload_dir, ignore_errors=True)


def discard_youtube_cookies(cookie_path):
    if cookie_path is None:
        return
    auth_dir = Path(cookie_path).resolve().parent
    runtime_root = YOUTUBE_COOKIE_RUNTIME_ROOT.resolve()
    if auth_dir.parent != runtime_root or not auth_dir.name.startswith(".muhtemel-ask-youtube-"):
        raise RuntimeError("Refusing to remove unexpected cookie directory")
    shutil.rmtree(auth_dir, ignore_errors=True)

## Download, probe and extract audio

In [ ]:
from src.download import YouTubeAuthenticationError, download_source
from src.media import extract_audio, save_source_metadata

try:
    download = download_source(
        SOURCE_URL, DIRS["source"], language="tr", force=FORCE_REBUILD,
        output_stem=EPISODE_NAME,
    )
except YouTubeAuthenticationError:
    print("YouTube requires browser verification; checking the saved Drive credential.")
    youtube_cookie_file = load_saved_youtube_cookies()
    if youtube_cookie_file is not None:
        try:
            download = download_source(
                SOURCE_URL, DIRS["source"], language="tr", force=FORCE_REBUILD,
                output_stem=EPISODE_NAME, cookies_file=youtube_cookie_file,
            )
        except YouTubeAuthenticationError:
            download = None
            print("Saved YouTube cookies were rejected; upload a fresh file.")
        finally:
            discard_youtube_cookies(youtube_cookie_file)
    else:
        download = None
    if download is None:
        youtube_cookie_file = upload_youtube_cookies()
        try:
            download = download_source(
                SOURCE_URL, DIRS["source"], language="tr", force=FORCE_REBUILD,
                output_stem=EPISODE_NAME, cookies_file=youtube_cookie_file,
            )
            save_youtube_cookies_to_drive(youtube_cookie_file)
        finally:
            discard_youtube_cookies(youtube_cookie_file)
source_metadata = save_source_metadata(
    download.video_path,
    DIRS["source"] / "source.media.json",
    original_url=SOURCE_URL,
    download_metadata=download.metadata,
)
audio = extract_audio(
    download.video_path,
    DIRS["prepare"],
    source_sha256=source_metadata["sha256"],
    force=FORCE_REBUILD,
)
print("Download resumed:", download.resumed)
print("Audio resumed:", audio.resumed)

## Turkish ASR and targeted verification

In [ ]:
from src.transcribe import TranscriptionConfig, transcribe_audio

transcription = transcribe_audio(
    audio.audio_path,
    DIRS["prepare"],
    config=TranscriptionConfig(
        model_name=WHISPER_MODEL,
        targeted_verification=TARGETED_VERIFICATION,
    ),
    captions_path=download.captions_path,
    canonical_names=tuple(names_config["canonical_names"]),
    religious_terms=tuple(x["source"] for x in religious_config["terms"]),
    force=FORCE_REBUILD,
)
print("Transcription resumed:", transcription.resumed)

## Lock segmentation, schema and translation batches

In [ ]:
from src.segment import SegmentationConfig, segment_transcription
from src.schema import build_episode_schema, save_schema
from src.batches import create_translation_pack

subtitle_cfg = series_config["subtitle"]
segmentation = segment_transcription(
    transcription.output_path,
    EPISODE,
    DIRS["prepare"],
    config=SegmentationConfig(
        schema_version=SCHEMA_VERSION,
        start_lead_ms=subtitle_cfg["start_lead_ms"],
        end_padding_ms=subtitle_cfg["end_padding_ms"],
        next_speech_guard_ms=subtitle_cfg["next_speech_guard_ms"],
        internal_gap_review_ms=subtitle_cfg["internal_gap_review_ms"],
        target_chars_per_line=subtitle_cfg["target_chars_per_line"],
        maximum_lines=subtitle_cfg["maximum_lines"],
    ),
    force=FORCE_REBUILD,
)
schema = build_episode_schema(
    segmentation.blocks, EPISODE, schema_version=SCHEMA_VERSION
)
schema_path = DIRS["prepare"] / "schema.json"
save_schema(schema_path, schema)

glossary = {
    "canonical_names": names_config["canonical_names"],
    "forbidden_name_variants": names_config.get("forbidden_variants", {}),
    "source_name_variants": names_config.get("source_variants", {}),
    "religious_terms": religious_config["terms"],
    "allah_only_semoga_is_invalid": religious_config.get("allah_only_semoga_is_invalid", True),
}
pack_path = DIRS["translation_input"] / f"{EPISODE_NAME}_TRANSLATION_PACK.zip"
manifest = create_translation_pack(
    schema,
    pack_path,
    glossary=glossary,
    instructions_path=SYSTEM_ROOT / "TRANSLATION_INSTRUCTIONS.md",
    batch_size=BATCH_SIZE,
    context_blocks=2,
)
print(f"Locked blocks: {len(schema['blocks']):,}")
print(f"Schema SHA-256: {schema['schema_sha256']}")
print(f"Translation pack ready: {pack_path}")